---
title: "The model behind the market demo"
subtitle: "One notebook, reusable Python, and executable examples"
---

This notebook is the literate source for `src/market_demo.py`. NBDev exports
the marked code cells to that module, while Quarto renders this same notebook
as a page in the project website. The examples and assertions below are also
run by NBDev, keeping the explanation close to the code it describes.

The simulated prices are synthetic and intended only to illustrate the
workflow; they are not a forecast or investment recommendation.

## Generate a reproducible price history

A geometric Brownian motion (GBM) gives a compact example of a reusable
function with input validation. It includes the initial price as day zero.

In [1]:
#| default_exp: market_demo
#| export: true
import numpy as np
import pandas as pd

TRADING_DAYS = 252

In [2]:
#| export: true
def generate_prices(
    n_days: int = 1260,
    initial_price: float = 100.0,
    drift: float = 0.07,
    volatility: float = 0.22,
    seed: int = 42,
) -> pd.Series:
    """Generate a reproducible daily GBM price series, including day zero."""
    if n_days < 1:
        raise ValueError("n_days must be at least 1")
    if not np.isfinite(initial_price) or initial_price <= 0:
        raise ValueError("initial_price must be finite and positive")
    if not np.isfinite(drift):
        raise ValueError("drift must be finite")
    if not np.isfinite(volatility) or volatility < 0:
        raise ValueError("volatility must be finite and non-negative")

    rng = np.random.default_rng(seed)
    dt = 1.0 / TRADING_DAYS
    shocks = rng.standard_normal(n_days)
    increments = (drift - 0.5 * volatility**2) * dt + volatility * np.sqrt(dt) * shocks
    values = initial_price * np.exp(np.concatenate(([0.0], np.cumsum(increments))))
    dates = pd.bdate_range("2020-01-01", periods=n_days + 1, name="Date")
    return pd.Series(values, index=dates, name="Close")

A fixed seed makes the example reproducible, and validation makes invalid
inputs fail explicitly rather than producing misleading output.

In [3]:
prices = generate_prices(n_days=5, seed=123)
assert len(prices) == 6
assert prices.iloc[0] == 100.0
assert (prices > 0).all()
prices

Date
2020-01-01    100.000000
2020-01-02     98.656488
2020-01-03     98.172752
2020-01-06     99.958931
2020-01-07    100.246223
2020-01-08    101.551325
Freq: B, Name: Close, dtype: float64

## Calculate log returns and estimate parameters

Consecutive log returns are additive over time. Their sample variance and
mean provide a simple annualized GBM calibration.

In [4]:
#| export: true
def log_returns(prices: pd.Series) -> pd.Series:
    """Calculate consecutive log returns for a positive price series."""
    if prices.empty or not np.isfinite(prices.to_numpy()).all() or (prices <= 0).any():
        raise ValueError("prices must be non-empty, finite, and strictly positive")
    returns = np.log(prices).diff().dropna()
    returns.name = "Daily log return"
    return returns

In [5]:
#| export: true
def estimate_parameters(returns: pd.Series) -> dict[str, float]:
    """Estimate annualized GBM drift and volatility from daily log returns."""
    clean_returns = returns.dropna()
    if len(clean_returns) < 2 or not np.isfinite(clean_returns).all():
        raise ValueError("at least two finite returns are required")

    daily_variance = float(clean_returns.var(ddof=1))
    return {
        "drift": float(clean_returns.mean() * TRADING_DAYS + 0.5 * daily_variance * TRADING_DAYS),
        "volatility": float(np.sqrt(daily_variance * TRADING_DAYS)),
    }

In [6]:
returns = log_returns(generate_prices(n_days=100, seed=3))
estimates = estimate_parameters(returns)
assert len(returns) == 100
assert np.isfinite(list(estimates.values())).all()
estimates

{'drift': -0.14213318752010448, 'volatility': 0.2357719414279167}

## Simulate paths

The simulator returns one column per path and includes the starting price
in its first row.

In [7]:
#| export: true
def simulate_gbm(
    initial_price: float,
    n_days: int,
    drift: float,
    volatility: float,
    n_paths: int = 250,
    seed: int = 7,
) -> pd.DataFrame:
    """Simulate daily GBM paths and return a day-by-path DataFrame."""
    if not np.isfinite(initial_price) or initial_price <= 0:
        raise ValueError("initial_price must be finite and positive")
    if n_days < 1 or n_paths < 1:
        raise ValueError("n_days and n_paths must be at least 1")
    if not np.isfinite(drift):
        raise ValueError("drift must be finite")
    if not np.isfinite(volatility) or volatility < 0:
        raise ValueError("volatility must be finite and non-negative")

    rng = np.random.default_rng(seed)
    dt = 1.0 / TRADING_DAYS
    shocks = rng.standard_normal((n_days, n_paths))
    increments = (drift - 0.5 * volatility**2) * dt + volatility * np.sqrt(dt) * shocks
    log_paths = np.vstack((np.zeros(n_paths), np.cumsum(increments, axis=0)))
    values = initial_price * np.exp(log_paths)
    return pd.DataFrame(
        values,
        index=pd.RangeIndex(n_days + 1, name="Trading day"),
        columns=[f"Path {i + 1}" for i in range(n_paths)],
    )

In [8]:
paths = simulate_gbm(
    initial_price=100.0,
    n_days=20,
    drift=estimates["drift"],
    volatility=estimates["volatility"],
    n_paths=8,
    seed=5,
)
assert paths.shape == (21, 8)
assert (paths.iloc[0] == 100.0).all()
assert np.isfinite(paths.to_numpy()).all()
paths.iloc[:5, :3]

,Path 1,Path 2,Path 3
Trading day,,,
0,100.000000,100.000000,100.000000
1,98.749407,97.986154,99.564646
2,99.786373,100.326719,99.901436
3,99.595227,98.541840,98.905357
4,100.135469,99.696654,96.455959


## What the notebook-first workflow adds

- **One source:** the exported Python functions and their explanations live
  together here rather than being maintained in separate files.
- **Executable documentation:** examples and checks run as notebook tests
  with `nbdev_test` and render as a Quarto web page.
- **Reusable code:** `nbdev_export` updates `src/market_demo.py`, which the
  existing report, dashboard, and unit tests already use.

The existing Quarto analysis and dashboard remain separate presentation
formats; this notebook is an experiment in using one literate source for
library code and its documentation.